# Final evaluation launcher — selected heuristics + external baselines

This notebook launches the **final thesis evaluation only**. It does **not** call the LLM, does **not** generate new heuristics, and does **not** modify the LLM loop pipeline.

Recommended workflow:

1. Edit the control panel.
2. Mount Drive.
3. Install only the final-evaluation dependencies.
4. Build a temporary runtime config.
5. Run a dry run.
6. Run a tiny smoke test.
7. Launch the full resumable evaluation.
8. Use the progress cells to monitor, stop, and resume.


## 1. Control panel

In [ ]:
# ============================================================
# FINAL EVALUATION CONTROL PANEL
# ============================================================

# -------------------------
# Repo / runtime
# -------------------------
ORG = "TM-HESSO-202526"
REPO = "llm-clustering-heuristics"
BRANCH = "main"
REFRESH_REPO_FROM_GITHUB = True   # Set True if you want Colab to clone a fresh copy.

# -------------------------
# Drive paths
# -------------------------
TM_DIR = "/content/drive/MyDrive/TM"
CLUSTER_ZIP_PATH = f"{TM_DIR}/cluster_tai.zip"

# Put Prof. Taillard's C++ zip or .cpp here. The runner compiles it with g++ -O2.
TAILLARD_CPP_SOURCE_PATH = "external/taillard_cpp/clustering_sphere.cpp"

# Optional references. Set to None if missing; raw objective values will still be reported.
SSE_REFERENCE_PATH = f"{TM_DIR}/kmeans.res"
PMEDIAN_REFERENCE_PATH = f"{TM_DIR}/kmeans.res"
RADIUS_FREE_REFERENCE_PATH = f"{TM_DIR}/generator_radius_reference_last_p.zip"
RADIUS_DATA_POINT_REFERENCE_PATH = f"{TM_DIR}/generator_radius_reference_last_p.zip"

ARTIFACT_DIR = f"{TM_DIR}/final-clustering-evaluation/final_eval_run"
INSTANCE_EXTRACT_DIR = "/content/final_eval_cluster_tai_extract"

# -------------------------
# Evaluation scope
# -------------------------
OBJECTIVES = ["sse", "pmedian", "radius"]
D_VALUES = [2, 3, 4]
P_VALUES = [15, 20, 25, 30, 40, 50, 70, 87, 100]
INSTANCE_IDS = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

# -------------------------
# Statistical protocol
# -------------------------
REPETITIONS = 30
TIMEOUT_S = 300
GLOBAL_SEED = 12345
CHECKPOINT_EVERY = 1
RESUME = True

# -------------------------
# Debug / launch controls
# -------------------------
DRY_RUN = False
SMOKE_TEST = True              # True = override to a tiny, fast test.
STOP_AFTER_JOBS = None         # Example: 20 to test resumability, None for no artificial stop.

# Smoke-test override. Used only when SMOKE_TEST=True.
SMOKE_OBJECTIVES = ["sse"]
SMOKE_D_VALUES = [2]
SMOKE_P_VALUES = [20]
SMOKE_INSTANCE_IDS = [0]
SMOKE_REPETITIONS = 2
SMOKE_TIMEOUT_S = 60



## 2. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
# ============================================================
# MACHINE / RUNTIME CHARACTERISTICS
# Save this before the final evaluation run
# ============================================================

from pathlib import Path
import platform
import subprocess
import json
import os
import sys
import time
import socket
import shutil

def run_cmd(cmd):
    try:
        out = subprocess.check_output(cmd, shell=True, stderr=subprocess.STDOUT, text=True)
        return out.strip()
    except Exception as e:
        return f"ERROR: {repr(e)}"

machine_info = {}

machine_info["timestamp_utc"] = time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime())
machine_info["hostname"] = socket.gethostname()
machine_info["platform"] = platform.platform()
machine_info["python_version"] = sys.version
machine_info["working_directory"] = os.getcwd()

# CPU / RAM
machine_info["cpu_model_lscpu"] = run_cmd("lscpu")
machine_info["cpu_model_proc"] = run_cmd("cat /proc/cpuinfo | grep 'model name' | head -1")
machine_info["cpu_core_count_os"] = os.cpu_count()
machine_info["memory_free_h"] = run_cmd("free -h")
machine_info["memory_proc_meminfo_head"] = run_cmd("head -20 /proc/meminfo")

# Disk
machine_info["disk_df_h"] = run_cmd("df -h")
machine_info["disk_usage_current_dir"] = run_cmd("du -sh . 2>/dev/null || true")

# GPU, if any
machine_info["nvidia_smi"] = run_cmd("nvidia-smi")
machine_info["gpu_lspci"] = run_cmd("lspci | grep -i 'vga\\|3d\\|nvidia' || true")

# OS / compiler / libraries
machine_info["gcc_version"] = run_cmd("g++ --version | head -1")
machine_info["pip_freeze_selected"] = run_cmd(
    "python - <<'PY'\n"
    "import importlib.metadata as md\n"
    "pkgs = ['numpy','pandas','scipy','scikit-learn','scikit-learn-extra','kmedoids','matplotlib']\n"
    "for p in pkgs:\n"
    "    try:\n"
    "        print(f'{p}=={md.version(p)}')\n"
    "    except Exception:\n"
    "        print(f'{p}: NOT INSTALLED')\n"
    "PY"
)

# Colab-specific hint
machine_info["colab_env"] = {
    "COLAB_GPU": os.environ.get("COLAB_GPU"),
    "COLAB_TPU_ADDR": os.environ.get("COLAB_TPU_ADDR"),
    "KAGGLE_KERNEL_RUN_TYPE": os.environ.get("KAGGLE_KERNEL_RUN_TYPE"),
}

print("=" * 80)
print("MACHINE / RUNTIME CHARACTERISTICS")
print("=" * 80)
for k, v in machine_info.items():
    print(f"\n--- {k} ---")
    print(v)

# Save into artifact directory
artifact_dir = Path(ARTIFACT_DIR + ("_smoke" if SMOKE_TEST else ""))
artifact_dir.mkdir(parents=True, exist_ok=True)

json_path = artifact_dir / "machine_info.json"
txt_path = artifact_dir / "machine_info.txt"

json_path.write_text(json.dumps(machine_info, indent=2), encoding="utf-8")

with txt_path.open("w", encoding="utf-8") as f:
    for k, v in machine_info.items():
        f.write(f"\n{'=' * 80}\n{k}\n{'=' * 80}\n")
        f.write(str(v))
        f.write("\n")

print("\nSaved machine info to:")
print(json_path)
print(txt_path)

## 3. Clone/enter repo and install final-evaluation dependencies

In [ ]:
repo_dir = f"/content/{REPO}"
repo_url = f"https://github.com/{ORG}/{REPO}.git"

%cd /content
!rm -rf "{repo_dir}"
!git clone --branch "{BRANCH}" "{repo_url}" "{repo_dir}"
%cd "{repo_dir}"
!ls


## 4. Build temporary runtime config from the control panel

In [ ]:
import yaml
from pathlib import Path

REPO_DIR = Path.cwd()
base_cfg_path = REPO_DIR / "configs" / "final_eval.yaml"
with open(base_cfg_path, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

objectives = SMOKE_OBJECTIVES if SMOKE_TEST else OBJECTIVES
d_values = SMOKE_D_VALUES if SMOKE_TEST else D_VALUES
p_values = SMOKE_P_VALUES if SMOKE_TEST else P_VALUES
instance_ids = SMOKE_INSTANCE_IDS if SMOKE_TEST else INSTANCE_IDS
reps = SMOKE_REPETITIONS if SMOKE_TEST else REPETITIONS
timeout_s = SMOKE_TIMEOUT_S if SMOKE_TEST else TIMEOUT_S

cfg["cluster_zip_path"] = CLUSTER_ZIP_PATH
cfg["selected_heuristics_dir"] = "experiments/selected_clustering_heuristics_final_by_objective"
cfg["artifact_dir"] = ARTIFACT_DIR + ("_smoke" if SMOKE_TEST else "")
cfg["instance_extract_dir"] = INSTANCE_EXTRACT_DIR
cfg["objectives"] = objectives
cfg["instance_filters"] = {
    "d_values": d_values,
    "p_values": p_values,
    "instance_ids": instance_ids,
    "min_n": None,
    "max_n": None,
}
cfg["repetitions"] = reps
cfg["timeout_s"] = timeout_s
cfg["global_seed"] = GLOBAL_SEED
cfg["resume"] = RESUME
cfg["checkpoint_every"] = CHECKPOINT_EVERY
cfg["taillard_cpp"]["source_path"] = TAILLARD_CPP_SOURCE_PATH

cfg["reference_tables"] = {
    "sse": SSE_REFERENCE_PATH,
    "pmedian": PMEDIAN_REFERENCE_PATH,
    # Run C free-center and data-point-center variants share the same generator/last-p reference.
    "radius_free": RADIUS_FREE_REFERENCE_PATH,
    "radius_data_point": RADIUS_DATA_POINT_REFERENCE_PATH,
    "radius": None,
}

# If you set a reference path to None in the control panel, keep it as null in YAML.
for k, v in list(cfg["reference_tables"].items()):
    if v is None or str(v).strip().lower() in {"", "none", "null"}:
        cfg["reference_tables"][k] = None

runtime_cfg_path = REPO_DIR / "configs" / "final_eval_runtime.yaml"
with open(runtime_cfg_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

print("Runtime config:", runtime_cfg_path)
print("Artifact dir:", cfg["artifact_dir"])
print("Objectives:", cfg["objectives"])
print("Instance filters:", cfg["instance_filters"])
print("Repetitions:", cfg["repetitions"])
print("Timeout per job:", cfg["timeout_s"])
print("Taillard C++ source:", cfg["taillard_cpp"]["source_path"])



## Evaluation transparency checks

These cells make the final-evaluation launcher explicit before anything expensive runs.

They show:

- where selected LLM heuristics are loaded from,
- which selected LLM heuristic files exist in the cloned repo,
- which external baselines are enabled in the runtime config,
- which methods were actually loaded by the runner after launch,
- current progress / last checkpointed jobs.

Important: `experiments/selected_clustering_heuristics_final_by_objective/` is **repo content**, not Google Drive content.  
It should resolve to `/content/llm-clustering-heuristics/experiments/selected_clustering_heuristics_final_by_objective/` after cloning the repo.



In [ ]:
from pathlib import Path
import yaml
import pandas as pd
from IPython.display import display

runtime_config_path = REPO_DIR / "configs" / "final_eval_runtime.yaml"

print("=== RUNTIME CONFIG / PATHS ===")
print("REPO_DIR:", REPO_DIR)
print("Runtime config:", runtime_config_path)
print("Runtime config exists:", runtime_config_path.exists())

if not runtime_config_path.exists():
    raise FileNotFoundError(
        f"Runtime config not found: {runtime_config_path}\n"
        "Run the config-building cell before this transparency check."
    )

with open(runtime_config_path, "r") as f:
    cfg = yaml.safe_load(f)

print("\n=== ACTIVE FINAL EVALUATION CONFIG ===")
print("Objectives:", cfg.get("objectives"))
print("Repetitions:", cfg.get("repetitions"))
print("Timeout:", cfg.get("timeout_s"))
print("Artifact dir:", cfg.get("artifact_dir"))
print("Cluster zip path:", cfg.get("cluster_zip_path"))
print("Taillard C++ source path:", (cfg.get("taillard_cpp") or {}).get("source_path"))
print("Selected heuristics dir, as configured:", cfg.get("selected_heuristics_dir"))
print("Reference tables:", cfg.get("reference_tables"))

print("\n=== INSTANCE FILTERS ===")
print(cfg.get("instance_filters"))

selected_dir = REPO_DIR / cfg.get(
    "selected_heuristics_dir",
    "experiments/selected_clustering_heuristics_final_by_objective"
)

print("\n=== SELECTED LLM HEURISTICS SOURCE ===")
print("Resolved local path:", selected_dir)
print("Exists:", selected_dir.exists())

if not selected_dir.exists():
    raise FileNotFoundError(
        f"Selected heuristics folder not found: {selected_dir}\n"
        "This folder is expected inside the cloned GitHub repo."
    )

objective_folders = {
    "sse_free": ("sse", "free", "SSE_free_centers"),
    "pmedian_data_point": ("pmedian", "snap_to_points", "P_MEDIAN_data_point_centers"),
    "radius_free": ("radius", "free", "RADIUS_VOLUME_free_centers"),
    "radius_data_point": ("radius", "snap_to_points", "RADIUS_VOLUME_data_point_centers"),
}

heuristic_rows = []

for variant, (objective, center_constraint, folder_name) in objective_folders.items():
    folder = selected_dir / folder_name
    py_files = sorted(folder.rglob("*.py")) if folder.exists() else []

    print(f"\n{variant} -> {folder}")
    print("objective:", objective, "| center_constraint:", center_constraint)
    print("exists:", folder.exists(), "| python files:", len(py_files))

    for py in py_files:
        info = py.parent / "INFO.txt"
        source_row = py.parent / "source_row.json"
        heuristic_rows.append({
            "type": "llm_selected",
            "objective": objective,
            "variant": variant,
            "center_constraint": center_constraint,
            "method_folder": py.parent.name,
            "python_file": py.name,
            "has_info_txt": info.exists(),
            "has_source_row_json": source_row.exists(),
            "path": str(py.relative_to(REPO_DIR)),
        })

heuristics_df = pd.DataFrame(heuristic_rows)

print("\n=== LLM HEURISTICS AVAILABLE IN THE REPO CLONE ===")
if heuristics_df.empty:
    raise RuntimeError("No selected LLM heuristic .py files were found.")
else:
    display(heuristics_df)
    print("\nCounts by objective/variant:")
    display(heuristics_df.groupby(["objective", "variant", "center_constraint"]).size().reset_index(name="num_llm_heuristics"))

baseline_rows = []

baselines = cfg.get("baselines", []) or []
if isinstance(baselines, dict):
    # Backward-compatible display for old nested configs.
    for objective, methods in baselines.items():
        for method_id, method_cfg in (methods or {}).items():
            method_cfg = method_cfg or {}
            if bool(method_cfg.get("enabled", True)):
                baseline_rows.append({
                    "type": "external_baseline",
                    "objective": objective,
                    "method_id": method_id,
                    "enabled": True,
                    "repetitions_override": method_cfg.get("repetitions"),
                    "center_constraint": method_cfg.get("center_constraint"),
                    "max_n": method_cfg.get("max_n"),
                })
else:
    enabled_objectives = set(cfg.get("objectives", []))
    for b in baselines:
        if not b.get("enabled", True):
            continue
        for objective in sorted((set(b.get("objectives", [])) or enabled_objectives) & enabled_objectives):
            baseline_rows.append({
                "type": "external_baseline",
                "objective": objective,
                "method_id": b.get("id"),
                "enabled": True,
                "method_type": b.get("type", "python"),
                "repetitions_override": b.get("repetitions"),
                "center_constraint": b.get("center_constraint"),
                "max_n": b.get("max_n"),
            })

baseline_df = pd.DataFrame(baseline_rows)

print("\n=== EXTERNAL BASELINES ENABLED IN THE CONFIG ===")
if baseline_df.empty:
    print("No external baselines enabled.")
else:
    display(baseline_df)
    print("\nCounts by objective:")
    display(baseline_df.groupby("objective").size().reset_index(name="num_baselines"))

print("\n=== EXPECTED METHOD COUNTS BY OBJECTIVE BEFORE REPEATS ===")
llm_counts = heuristics_df.groupby("objective").size().reset_index(name="llm_selected")
base_counts = baseline_df.groupby("objective").size().reset_index(name="baselines") if not baseline_df.empty else pd.DataFrame(columns=["objective", "baselines"])
display(llm_counts.merge(base_counts, on="objective", how="outer").fillna(0))



## 5. Check required files

In [ ]:
from pathlib import Path

paths = {
    "cluster_zip_path": CLUSTER_ZIP_PATH,
    "taillard_cpp_source_path": TAILLARD_CPP_SOURCE_PATH,
    "sse_reference": SSE_REFERENCE_PATH,
    "pmedian_reference": PMEDIAN_REFERENCE_PATH,
    "radius_free_reference": RADIUS_FREE_REFERENCE_PATH,
    "radius_data_point_reference": RADIUS_DATA_POINT_REFERENCE_PATH,
}
for name, path in paths.items():
    if path is None:
        print(f"{name}: null")
    else:
        p = Path(path)
        print(f"{name}: {'OK' if p.exists() else 'MISSING'} -> {p}")

print("\nSelected heuristics folders:")
!find experiments/selected_clustering_heuristics_final_by_objective -maxdepth 3 -type f -name '*.py' | head -40



## 6. Dry run / method manifest

In [ ]:
import subprocess, sys

cmd = [sys.executable, "scripts/run_final_evaluation.py", "--config", str(runtime_cfg_path), "--dry-run"]
print(" ".join(cmd))
subprocess.run(cmd, check=True)


## 7. Launch evaluation with live progress

In [ ]:
import subprocess, sys, signal, os, time
from pathlib import Path

cmd = [sys.executable, "scripts/run_final_evaluation.py", "--config", str(runtime_cfg_path)]
if not RESUME:
    cmd.append("--no-resume")
if STOP_AFTER_JOBS is not None:
    cmd += ["--stop-after-jobs", str(STOP_AFTER_JOBS)]

print("Launching:", " ".join(cmd))
print("Stop safely with the notebook stop button / KeyboardInterrupt. Relaunch this same cell to resume.")

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
except KeyboardInterrupt:
    print("\nKeyboardInterrupt: forwarding SIGINT to evaluation process so it can checkpoint...")
    proc.send_signal(signal.SIGINT)
    try:
        proc.wait(timeout=30)
    except subprocess.TimeoutExpired:
        print("Process did not stop after SIGINT; terminating.")
        proc.terminate()
        proc.wait(timeout=10)
finally:
    rc = proc.poll()
    print("\nEvaluation process return code:", rc)


## Post-launch visibility

Run these cells after launching the evaluation to see which methods were actually loaded and what progress has been checkpointed.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

artifact_dir = Path(ARTIFACT_DIR + ("_smoke" if SMOKE_TEST else ""))
manifest_path = artifact_dir / "method_manifest.csv"

print("=== ACTUALLY LOADED METHODS AFTER RUNNER STARTED ===")
print("Artifact dir:", artifact_dir)
print("Method manifest:", manifest_path)
print("Exists:", manifest_path.exists())

if manifest_path.exists() and manifest_path.stat().st_size > 0:
    manifest = pd.read_csv(manifest_path)
    print("Loaded methods:", len(manifest))
    display(manifest)

    print("\nCounts:")
    display(
        manifest.groupby(["objective", "method_group"])
        .size()
        .reset_index(name="num_methods")
    )
else:
    print("No method_manifest.csv yet. Launch the evaluation cell first.")


In [ ]:
from pathlib import Path
import pandas as pd
import json
from IPython.display import display

artifact_dir = Path(ARTIFACT_DIR + ("_smoke" if SMOKE_TEST else ""))

progress_path = artifact_dir / "progress_state.json"
checkpoint_path = artifact_dir / "raw_runs_checkpoint.csv"
log_path = artifact_dir / "logs" / "progress.log"

print("=== LIVE / CHECKPOINT PROGRESS ===")
print("Artifact dir:", artifact_dir)

print("\n--- progress_state.json ---")
if progress_path.exists():
    try:
        print(json.dumps(json.loads(progress_path.read_text()), indent=2)[:5000])
    except Exception as e:
        print("Could not parse progress_state.json:", repr(e))
        print(progress_path.read_text(errors="replace")[:2000])
else:
    print("No progress_state.json yet.")

print("\n--- last checkpointed runs ---")
if checkpoint_path.exists() and checkpoint_path.stat().st_size > 0:
    df = pd.read_csv(checkpoint_path)
    print("Rows completed:", len(df))
    cols = [
        "objective", "method_group", "method_id", "instance", "rep",
        "success", "timeout", "runtime_s", "objective_value", "gap_pct", "error"
    ]
    cols = [c for c in cols if c in df.columns]
    display(df.tail(30)[cols])
else:
    print("No raw_runs_checkpoint.csv yet.")

print("\n--- last progress log lines ---")
if log_path.exists():
    lines = log_path.read_text(errors="replace").splitlines()
    print("\n".join(lines[-60:]))
else:
    print("No progress.log yet.")


## Safe result display

This version does not crash when `complexity_fit.csv` is empty during small smoke tests.


In [ ]:
from pathlib import Path
import pandas as pd
from pandas.errors import EmptyDataError
from IPython.display import display, Image

artifact_dir = Path(ARTIFACT_DIR + ("_smoke" if SMOKE_TEST else ""))

print("Artifact dir:", artifact_dir)

for name in [
    "method_summary.csv",
    "instance_summary.csv",
    "complexity_fit.csv",
    "complexity_fit_points.csv",
    "center_repair_warning_counts.json",
]:
    p = artifact_dir / name
    print("\n", name, "->", "OK" if p.exists() else "missing")

    if not p.exists():
        continue

    if p.suffix == ".csv":
        if p.stat().st_size == 0:
            print("File exists but is empty. This is normal when there are not enough data points, e.g. smoke complexity with only one n.")
            continue

        try:
            df = pd.read_csv(p)
            if df.empty:
                print("CSV has columns but no rows.")
            else:
                display(df.head(30))
        except EmptyDataError:
            print("CSV exists but has no parseable columns. Usually means no data was available for this summary.")
    else:
        print(p.read_text(errors="replace")[:3000])

plot_dir = artifact_dir / "complexity_plots"
if plot_dir.exists():
    pngs = sorted(plot_dir.glob("*.png"))
    if not pngs:
        print("\ncomplexity_plots exists but contains no PNG yet.")
    for png in pngs:
        print("\n", png.name)
        display(Image(filename=str(png)))
else:
    print("\nNo complexity_plots directory yet.")

zip_path = str(artifact_dir) + ".zip"
print("\nArtifact zip:", zip_path, "exists=", Path(zip_path).exists())


## 8. Read live progress / checkpoint summaries

In [ ]:
import json
from pathlib import Path
import pandas as pd

artifact_dir = Path(ARTIFACT_DIR + ("_smoke" if SMOKE_TEST else ""))
progress_file = artifact_dir / "progress_state.json"
checkpoint_file = artifact_dir / "raw_runs_checkpoint.csv"

if progress_file.exists():
    print(json.dumps(json.loads(progress_file.read_text()), indent=2))
else:
    print("No progress_state.json yet:", progress_file)

if checkpoint_file.exists():
    df = pd.read_csv(checkpoint_file)
    print("\nCheckpoint rows:", len(df))
    display(df.tail(20))
    print("\nSuccess/timeout by objective/method:")
    display(df.groupby(["objective", "method_id"]).agg(rows=("success", "count"), success_rate=("success", "mean"), timeout_rate=("timeout", "mean")).reset_index().tail(50))
else:
    print("No checkpoint yet:", checkpoint_file)


## 9. Inspect final summaries and complexity plots

In [ ]:
from pathlib import Path
import pandas as pd
from pandas.errors import EmptyDataError
from IPython.display import display, Image
import shutil
import time

# ============================================================
# Inspect artifacts + download artifact zip to local PC
# ============================================================

DOWNLOAD_TO_LOCAL_PC = True
FORCE_RECREATE_ZIP = True

artifact_dir = Path(ARTIFACT_DIR + ("_smoke" if SMOKE_TEST else ""))
print("Artifact directory:", artifact_dir)
print("Artifact directory exists:", artifact_dir.exists())

if not artifact_dir.exists():
    raise FileNotFoundError(f"Artifact directory does not exist: {artifact_dir}")

# ------------------------------------------------------------
# Show important artifact files
# ------------------------------------------------------------
for name in [
    "method_manifest.csv",
    "raw_runs.csv",
    "raw_runs_checkpoint.csv",
    "progress_state.json",
    "method_summary.csv",
    "instance_summary.csv",
    "complexity_fit.csv",
    "complexity_fit_points.csv",
    "center_repair_warning_counts.json",
]:
    p = artifact_dir / name
    print("\n", name, "->", "OK" if p.exists() else "missing")

    if not p.exists():
        continue

    if p.suffix == ".csv":
        if p.stat().st_size == 0:
            print("File exists but is empty. This is normal if there are not enough data points yet.")
            continue

        try:
            df = pd.read_csv(p)
            print("shape:", df.shape)
            if df.empty:
                print("CSV has columns but no rows.")
            else:
                display(df.head(30))
        except EmptyDataError:
            print("CSV exists but has no parseable columns. Usually means no data was available for this summary.")
    else:
        print(p.read_text(encoding="utf-8", errors="ignore")[:2000])

# ------------------------------------------------------------
# Show complexity plots if available
# ------------------------------------------------------------
plot_dir = artifact_dir / "complexity_plots"
if plot_dir.exists():
    pngs = sorted(plot_dir.glob("*.png"))
    if not pngs:
        print("\ncomplexity_plots exists but contains no PNG yet.")

    for png in pngs[:20]:
        print("\n", png.name)
        display(Image(filename=str(png)))

    if len(pngs) > 20:
        print(f"\nDisplayed first 20 PNGs out of {len(pngs)}.")
else:
    print("\nNo complexity_plots directory yet.")

# ------------------------------------------------------------
# Create or recreate zip
# ------------------------------------------------------------
zip_path = Path(str(artifact_dir) + ".zip")

if FORCE_RECREATE_ZIP or not zip_path.exists():
    if zip_path.exists():
        print("\nRemoving old zip:", zip_path)
        zip_path.unlink()

    print("\nCreating zip from artifact directory...")
    created = shutil.make_archive(
        base_name=str(artifact_dir),
        format="zip",
        root_dir=str(artifact_dir.parent),
        base_dir=artifact_dir.name,
    )
    zip_path = Path(created)

print("\nArtifact zip:", zip_path)
print("Zip exists:", zip_path.exists())
print("Zip size MB:", round(zip_path.stat().st_size / (1024 * 1024), 2))

# ------------------------------------------------------------
# Download to local PC
# ------------------------------------------------------------
if DOWNLOAD_TO_LOCAL_PC:
    try:
        from google.colab import files

        print("\nStarting download to your local PC/browser...")
        print("Keep this tab open until the browser download starts/finishes.")
        files.download(str(zip_path))
        print("\nDownload command sent.")
    except Exception as e:
        print("\nCould not trigger automatic browser download.")
        print("Error:", repr(e))
        print("The zip is still saved at:")
        print(zip_path)
else:
    print("\nDOWNLOAD_TO_LOCAL_PC=False, so no local download was triggered.")